In [8]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("PINECONE_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [2]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [4]:
from pinecone import Pinecone, ServerlessSpec

index_name = "hybrid-search-pinecone"

pc = Pinecone(api_key=api_key)

In [6]:
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,  # Dimensions for Dense vectors
        metric="dotproduct",  # Sparse value support only for dotproduct
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

In [7]:
index = pc.Index(index_name)
index

c:\ProgramData\miniconda3\envs\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Vector Embedding & Sparse Matrix


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6061.64it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
from pinecone_text.sparse import BM25Encoder

bm25_encoder = BM25Encoder().default()

bm25_encoder

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Prashant\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Prashant\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [14]:
sentences = [
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans",
]

# TF-IDF Applied

bm25_encoder.fit(sentences)

bm25_encoder.dump("bm25_values.json")

bm25_encoder = BM25Encoder().load("bm25_values.json")

100%|██████████| 3/3 [00:00<?, ?it/s]


In [19]:
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings, sparse_encoder=bm25_encoder, index=index
)

In [20]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x000001B4C18BFB60>, index=<pinecone.db_data.index.Index object at 0x000001B41760ECC0>)

In [21]:
retriever.add_texts(
    [
        "In 2023, I visited Paris",
        "In 2022, I visited New York",
        "In 2021, I visited New Orleans",
    ]
)

100%|██████████| 1/1 [00:01<00:00,  1.47s/it]


In [23]:
retriever.invoke("What city did I visit recent?")

[Document(metadata={'score': 0.289176583}, page_content='In 2021, I visited New Orleans'),
 Document(metadata={'score': 0.246345654}, page_content='In 2022, I visited New York'),
 Document(metadata={'score': 0.217880458}, page_content='In 2023, I visited Paris')]